# CICIoMT2024 · Full SVM on Google Colab — 4 FW/FS configurations

**Notebook riêng cho `baseline_svm` và `mi_rfe_svm` của repository Stage 7.3B.** Giữ nguyên `ciciomt24_training.py` và thuật toán/siêu tham số trong Git (SVC RBF, `C=10`, `max_iter=20000`, `probability=True` ở baseline, `class_weight='balanced'` ở compact, `cache_size=10000`). Không lấy mẫu để thay thế full training.

**Cách dùng:** Chạy các cell **theo thứ tự từng cell**, không chọn *Runtime → Run all*. Mỗi lượt **chọn đúng một** trong bốn cấu hình FW256/FS8, FW256/FS16, FW512/FS8, FW512/FS16 và một nhánh SVM. Lặp lại notebook cho cấu hình khác; mỗi run cần probe, approval và xác nhận riêng. Không chạy nhiều SVM full song song.

**Về hiệu năng:** Dữ liệu cửa sổ được *sao chép có xác minh* từ Drive sang SSD `/content` (không chỉnh sửa giá trị). Metadata, source hash và dataset identity được kiểm tra trước chạy. Tiết kiệm I/O không bảo đảm giảm nhiều thời gian `SVC.fit()` với hàng triệu mẫu.

**Cảnh báo khoa học:** nhãn `SCENARIO_PROXY`, chưa xác nhận ground truth từng flow hoặc tính độc lập capture/session. Test chỉ đánh giá **một lần cuối** trong pipeline chính thức. Bằng chứng probe/approval nháp không phải phê duyệt full run.
**Bản tích hợp từ FIXED_V3:** sửa Path khi copytree, xác minh SHA256 SSD, probe dự phòng, dung lượng tái sử dụng và hậu kiểm supervisor. Cell 3 chỉ chọn commit source đã duyệt khi clone mới; checkout đã có được giữ nguyên. Không mặc định cấp quyền training.


## 1. Chọn experiment/config và đường dẫn

Dùng source Stage 7.3B tại commit đã đối chiếu. Cell 3 chọn commit này trong checkout mới để việc push notebook lên main không làm lệch source huấn luyện. Checkout đã tồn tại phải sạch và đúng commit; nếu không, notebook dừng. `USE_SSD=True` cho phép staging ~20 GiB CSV lên `/content`; cần đủ dung lượng cho kết quả/cache huấn luyện.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

GITHUB_URL = 'https://github.com/kasum00/RFforNetwork.git'
EXPECTED_COMMIT = 'c93eab87395edaf8fd141b29ee9914df066a2e7e'
REPO = Path('/content/RFforNetwork_svm4_c93eab87395e')
DRIVE_ROOT = Path('/content/drive/MyDrive/COMPACT-MI-RFE')
DRIVE_WINDOWS = Path('/content/drive/MyDrive/WINDOWS_CPLUS')
DRIVE_MANIFEST = Path('/content/drive/MyDrive/CSV_SPLIT_CPLUS')
SSD_ROOT = Path('/content/svm_data')
SSD_WINDOWS = SSD_ROOT / 'WINDOWS_CPLUS'
SSD_MANIFEST = SSD_ROOT / 'CSV_SPLIT_CPLUS'
LOCAL_RUNS = Path('/content/stage73b_runs')

# ===== CHỈNH BA DÒNG NÀY CHO MỖI RUN =====
EXPERIMENT = 'baseline_svm'          # 'baseline_svm' hoặc 'mi_rfe_svm'
CONFIG = 'FW_512_FS_16'              # FW_256_FS_8 | FW_256_FS_16 | FW_512_FS_8 | FW_512_FS_16
RUN_ID = 'official01'
# ========================================
USE_SSD = True
CONFIG_SHORT = CONFIG.replace("FW_", "FW").replace("_FS_", "_FS")
RUN_NAME = f'{EXPERIMENT}_{CONFIG_SHORT}_{RUN_ID}'
assert EXPERIMENT in ('baseline_svm', 'mi_rfe_svm')
assert CONFIG in ('FW_256_FS_8','FW_256_FS_16','FW_512_FS_8','FW_512_FS_16')
assert RUN_ID and all(ch.isascii() and (ch.isalnum() or ch in '-_') for ch in RUN_ID)
WINDOWS = SSD_WINDOWS if USE_SSD else DRIVE_WINDOWS
APPROVAL = DRIVE_ROOT/'approvals'/f'{RUN_NAME}.json'
UNIT_EVIDENCE = DRIVE_ROOT/'environment'/f'stage73b_unit_tests_{EXPECTED_COMMIT[:12]}.json'
GIB = 1024**3

def sha256_file(p: Path):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for buf in iter(lambda: f.read(8*1024*1024), b''):
            h.update(buf)
    return h.hexdigest()

print(json.dumps({'experiment':EXPERIMENT,'config':CONFIG,'run_name':RUN_NAME,
                  'windows_selected':str(WINDOWS),'official_train':'DISABLED'}, indent=2))

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
for p in [DRIVE_ROOT, DRIVE_WINDOWS, DRIVE_MANIFEST]:
    if not p.is_dir(): raise FileNotFoundError(f'Missing Drive folder: {p}')
for name in ('split_manifest.csv','validation.json'):
    if not (DRIVE_MANIFEST/name).is_file(): raise FileNotFoundError(DRIVE_MANIFEST/name)
print('Drive + manifest: PASS')

## 3. Clone/kiểm tra chính xác code gốc

Khi thư mục chưa tồn tại, clone repository rồi chọn đúng `EXPECTED_COMMIT` bằng detached checkout. Không pull/reset/vá hoặc chuyển commit của checkout đã tồn tại; Stage 7.3B buộc Git sạch và danh tính chính xác.

In [ ]:
if not REPO.exists():
    subprocess.run(['git','clone',GITHUB_URL,str(REPO)],check=True)
    # Pin approved scientific source only in the checkout just created.
    subprocess.run(['git','-C',str(REPO),'switch','--detach',EXPECTED_COMMIT],check=True)
if not (REPO/'.git').is_dir():
    raise RuntimeError(f'{REPO} exists, but is not a Git checkout')
head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
dirty = subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
if dirty: raise RuntimeError('Repository is dirty; do not modify code under an existing approval')
if head != EXPECTED_COMMIT:
    raise RuntimeError(f'Git commit mismatch: {head}; expected {EXPECTED_COMMIT}. Update/push correct repo first; no auto-checkout.')
required = ('huyen_code/ciciomt24_training.py','huyen_code/ciciomt24_sliding_window.py',
            'colab/dataset_validator.py','colab/experiment_launcher.py','colab/svm_run_authorization.py',
            'colab/svm_official_supervisor.py','colab/setup_colab.py',
            'huyen_code/ciciomt24_svm_probe.py')
missing = [p for p in required if not (REPO/p).is_file()]
if missing: raise FileNotFoundError('Missing Stage 7.3B code: '+', '.join(missing))
for p in [REPO, REPO/'huyen_code']:
    if str(p) not in sys.path: sys.path.insert(0,str(p))
print('Code checkout PASS:', head)

## 4. Kiểm tra thư viện khoa học (không âm thầm đổi môi trường)

Đã xác nhận trên Colab của em: NumPy `2.2.6`, pandas `2.3.3`, SciPy `1.15.3`, sklearn `1.7.2`, joblib `1.5.3`. Nếu thiếu, bật cờ cài đặt phía dưới và **restart session**, sau đó chạy lại từ cell 1. `pip check` có thể báo pandas khác dependency của `google-colab`; cần cân nhắc cảnh báo riêng.

In [ ]:
from colab.setup_colab import runtime_info
INSTALL_SCIENTIFIC_PROFILE = False  # Chỉ đổi thành True nếu profile hiện tại không PASS
if INSTALL_SCIENTIFIC_PROFILE:
    subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:',
                    'numpy==2.2.6','pandas==2.3.3','scipy==1.15.3',
                    'scikit-learn==1.7.2','joblib==1.5.3'],check=True)
    raise RuntimeError('Installation finished: RESTART COLAB RUNTIME, then run again from cell 1')
info=runtime_info()
print('Selected scientific profile matches:',info.get('scientific_versions_match_selected_profile'))
print('Installed packages:', info.get('packages') or info.get('versions'))
if not info.get('scientific_versions_match_selected_profile'):
    raise RuntimeError('Scientific environment mismatch. Review target versions before enabling INSTALL_SCIENTIFIC_PROFILE')

## 5. Kiểm tra tồn kho dữ liệu cửa sổ & dung lượng SSD

Không đọc dòng Test, không fit. Tất cả bốn cấu hình nằm trong cùng `WINDOWS_CPLUS`. Khi dữ liệu SSD đã có, chỉ kiểm tra dung lượng dự phòng 35 GiB; **không tính lại dung lượng của toàn bộ CSV đã sao chép**. Cell 6 vẫn bắt buộc kiểm tra SHA256 của dữ liệu cũ và dừng nếu thiếu/hỏng.

In [ ]:
import psutil
import ciciomt24_training as training
source_store=training.WindowStore(DRIVE_WINDOWS, split_root=DRIVE_MANIFEST)
source_files = sorted(set(f.path for files in source_store.files.values() for f in files))
csv_bytes = sum(p.stat().st_size for p in source_files)
other_bytes = sum(p.stat().st_size for p in DRIVE_WINDOWS.rglob('*') if p.is_file() and p not in set(source_files))
free = shutil.disk_usage('/content').free
print('CSV files:',len(source_files),'CSV GiB:',round(csv_bytes/GIB,2),
      'Other data GiB:',round(other_bytes/GIB,2),'Local disk free GiB:',round(free/GIB,2))
for config in training.CONFIGS:
    f=source_store.files[config]
    print(config,{role:sum(x.count for x in f if x.role==role) for role in ('train','val','test')})
print('SVC baseline params:',training.BASELINE_SVM_PARAMS)
print('SVC compact params:',training.COMPACT_SVM_PARAMS)
assert training.BASELINE_SVM_PARAMS['kernel']=='rbf' and training.BASELINE_SVM_PARAMS['C']==10.0
assert training.COMPACT_SVM_PARAMS['kernel']=='rbf' and training.COMPACT_SVM_PARAMS['C']==10.0
# Existing SSD staging consumes disk already. Only reserve space for a new copy.
# Cell 6 independently fails closed when existing staging/checksums are missing or corrupt.
copy_required = 0 if SSD_WINDOWS.exists() else csv_bytes + other_bytes
print('Additional SSD copy required GiB:', round(copy_required/GIB, 2))
if USE_SSD and free < copy_required + 35*GIB:
    raise RuntimeError('Not enough SSD headroom for verified staging plus training/cache. Do not start copying')
if USE_SSD and SSD_WINDOWS.exists():
    print('Existing staged WINDOWS detected; Cell 6 must verify every SHA256 before use')
print('SOURCE DATA INVENTORY: PASS')

## 6. Copy SSD và xác minh **toàn bộ SHA256** (fail-closed)

Cell 6 sửa lỗi `shutil.copytree` truyền chuỗi vào `verified_copy`; ghi manifest SHA256 của **từng file đã copy** tại SSD và Drive. **Mỗi lần tái sử dụng SSD, bắt buộc hash lại toàn bộ file SSD và đối chiếu với manifest.** Không có chế độ bỏ qua kiểm tra nội dung. Khi manifest thiếu, sai hoặc có `.copying`, cell dừng và yêu cầu kiểm tra/khôi phục; không tự ghi đè dữ liệu.

Dữ liệu nguồn Drive được hash trong lần copy đầu tiên. Để phát hiện nguồn Drive bị chỉnh nội dung sau đó (kể cả cùng kích thước), bật `VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE=True`, sẽ tốn thêm thời gian đọc lại toàn bộ Drive. Mặc định luôn kiểm tra SHA256 SSD, checksum summary + manifest nguồn và kích thước file nguồn.

In [ ]:
# CELL 6 — STAGE DATA + CHECKSUM MANIFEST, NOT MODEL TRAINING
STAGE_DATA_NOW = True
VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE = False  # True => hash toàn bộ CSV gốc Drive ở mỗi lần tái sử dụng
STAGING_SCHEMA = 'stage73b.ssd.file-sha256.v1'
LOCAL_SHA_MANIFEST = SSD_ROOT / 'ssd_file_sha256_manifest.json'
DRIVE_SHA_MANIFEST = DRIVE_ROOT / 'environment' / 'ciciomt24_windows_cplus_ssd_file_sha256_manifest.json'

# Inventories are relative paths, never absolute destinations; no Test rows are parsed.
source_items = []
for p in sorted(DRIVE_WINDOWS.rglob('*')):
    if p.is_file():
        source_items.append(('windows', p.relative_to(DRIVE_WINDOWS).as_posix(), p))
for name in ('split_manifest.csv', 'validation.json'):
    source_items.append(('manifest', name, DRIVE_MANIFEST / name))
source_lookup = {(area, rel): p for area, rel, p in source_items}
assert len(source_lookup) == len(source_items), 'Duplicate source relative paths'
assert set(source_files).issubset({p for area, rel, p in source_items if area == 'windows'})

def staging_path(area, rel):
    if not rel or Path(rel).is_absolute() or '..' in Path(rel).parts:
        raise ValueError(f'Unsafe relative path: {rel!r}')
    return (SSD_WINDOWS if area == 'windows' else SSD_MANIFEST) / rel

def build_staging_manifest(sha_by_file):
    rows = []
    for area, rel, p in source_items:
        key = (area, rel)
        if key not in sha_by_file:
            raise RuntimeError(f'Source SHA256 not captured during copy: {key}')
        rows.append({'area': area, 'path': rel, 'bytes': p.stat().st_size,
                     'sha256': sha_by_file[key]})
    return {'schema': STAGING_SCHEMA, 'files': rows,
            'source_window_summary_sha256': sha_by_file[('windows', 'window_run_summary.json')],
            'source_split_manifest_sha256': sha_by_file[('manifest', 'split_manifest.csv')],
            'source_validation_sha256': sha_by_file[('manifest', 'validation.json')]}

def verify_staging_contents(payload):
    if not isinstance(payload, dict) or payload.get('schema') != STAGING_SCHEMA:
        raise RuntimeError('Staging SHA256 manifest schema mismatch')
    records = payload.get('files')
    if not isinstance(records, list) or len(records) != len(source_items):
        raise RuntimeError('Staging manifest file count mismatch')
    indexed = {}
    for row in records:
        if not isinstance(row, dict) or not isinstance(row.get('sha256'), str):
            raise RuntimeError('Invalid staging manifest row')
        key = (row.get('area'), row.get('path'))
        if key in indexed:
            raise RuntimeError(f'Duplicate staging manifest key: {key}')
        indexed[key] = row
    if set(indexed) != set(source_lookup):
        raise RuntimeError('Staging manifest source inventory mismatch')
    actual_windows = {p.relative_to(SSD_WINDOWS).as_posix()
                      for p in SSD_WINDOWS.rglob('*') if p.is_file()}
    manifest_windows = {rel for area, rel in indexed if area == 'windows'}
    if actual_windows != manifest_windows:
        raise RuntimeError('Staged WINDOWS contains missing/extra files')
    for i, (area, rel, source) in enumerate(source_items, 1):
        row = indexed[(area, rel)]
        dst = staging_path(area, rel)
        size = row.get('bytes')
        if not isinstance(size, int) or size < 0 or not dst.is_file():
            raise RuntimeError(f'Missing/invalid SSD file: {dst}')
        if dst.stat().st_size != size or source.stat().st_size != size:
            raise RuntimeError(f'File size mismatch (Drive vs SSD vs manifest): {rel}')
        if sha256_file(dst) != row['sha256']:
            raise RuntimeError(f'SSD SHA256 MISMATCH: {dst}')
        if VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE and sha256_file(source) != row['sha256']:
            raise RuntimeError(f'Drive source SHA256 changed: {source}')
        if i % 40 == 0 or i == len(source_items):
            print(f'Checksum verified: {i}/{len(source_items)}', flush=True)
    for area, rel, recorded_key in (
        ('windows', 'window_run_summary.json', 'source_window_summary_sha256'),
        ('manifest', 'split_manifest.csv', 'source_split_manifest_sha256'),
        ('manifest', 'validation.json', 'source_validation_sha256')
    ):
        if indexed[(area, rel)]['sha256'] != payload.get(recorded_key):
            raise RuntimeError(f'Critical source metadata checksum differs: {rel}')
        if sha256_file(source_lookup[(area, rel)]) != payload[recorded_key]:
            raise RuntimeError(f'Critical Drive metadata changed: {rel}')
    return len(records)

# copytree(copy_function=...) supplies STR, not Path. Record the digest of source stream.
copy_hashes = {}
def verified_copy(source, target):
    source = Path(source)  # P1: required for shutil.copytree
    target = Path(target)  # P1: required for shutil.copytree
    target.parent.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    with source.open('rb') as reader, target.open('xb') as writer:
        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
            writer.write(chunk)
    shutil.copystat(source, target)
    if target.stat().st_size != source.stat().st_size or sha256_file(target) != digest.hexdigest():
        raise OSError(f'Copy verification failed: {source} -> {target}')
    try:
        key = ('windows', source.relative_to(DRIVE_WINDOWS).as_posix())
    except ValueError:
        key = ('manifest', source.relative_to(DRIVE_MANIFEST).as_posix())
    if key in copy_hashes:
        raise RuntimeError(f'Duplicate copy record: {key}')
    copy_hashes[key] = digest.hexdigest()
    return str(target)

if USE_SSD:
    if not STAGE_DATA_NOW:
        raise RuntimeError('USE_SSD=True but STAGE_DATA_NOW=False: no verified SSD is available')
    SSD_ROOT.mkdir(parents=True, exist_ok=True)
    temp_win = SSD_ROOT / 'WINDOWS_CPLUS.copying'
    if temp_win.exists():
        raise FileExistsError(f'Interrupted staging requires manual inspection: {temp_win}')
    if not SSD_WINDOWS.exists():
        if LOCAL_SHA_MANIFEST.exists():
            raise RuntimeError('SHA256 manifest exists without SSD_WINDOWS; inspect state manually')
        if SSD_MANIFEST.exists() and any(SSD_MANIFEST.rglob('*')):
            raise RuntimeError('SSD manifest directory already present without staged windows; inspect state')
        start = time.monotonic()
        shutil.copytree(DRIVE_WINDOWS, temp_win, copy_function=verified_copy)
        for name in ('split_manifest.csv', 'validation.json'):
            verified_copy(DRIVE_MANIFEST / name, SSD_MANIFEST / name)
        SSD_WINDOWS.parent.mkdir(parents=True, exist_ok=True)
        temp_win.rename(SSD_WINDOWS)
        staged_payload = build_staging_manifest(copy_hashes)
        # Persist a source-anchored manifest. Existing Drive evidence is never replaced.
        DRIVE_SHA_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
        if DRIVE_SHA_MANIFEST.is_file():
            known = json.loads(DRIVE_SHA_MANIFEST.read_text(encoding='utf-8'))
            if known != staged_payload:
                raise RuntimeError('Existing Drive checksum manifest differs; do NOT overwrite')
        else:
            with DRIVE_SHA_MANIFEST.open('x', encoding='utf-8') as fh:
                json.dump(staged_payload, fh, indent=2, ensure_ascii=False)
        with LOCAL_SHA_MANIFEST.open('x', encoding='utf-8') as fh:
            json.dump(staged_payload, fh, indent=2, ensure_ascii=False)
        print('Copied and checksummed source in', round(time.monotonic()-start, 1), 'seconds')
    else:
        print('Existing SSD data detected: full SHA256 validation is mandatory')
    if not LOCAL_SHA_MANIFEST.is_file() or not DRIVE_SHA_MANIFEST.is_file():
        raise RuntimeError('SHA256 manifest missing. Staged data cannot be trusted/reused')
    local_checksum = json.loads(LOCAL_SHA_MANIFEST.read_text(encoding='utf-8'))
    drive_checksum = json.loads(DRIVE_SHA_MANIFEST.read_text(encoding='utf-8'))
    if local_checksum != drive_checksum:
        raise RuntimeError('Local SHA256 manifest differs from immutable Drive manifest')
    verified_count = verify_staging_contents(drive_checksum)
    staged_store = training.WindowStore(SSD_WINDOWS, split_root=SSD_MANIFEST)
    for config in training.CONFIGS:
        left, right = source_store.files[config], staged_store.files[config]
        if len(left) != len(right) or sum(f.count for f in left) != sum(f.count for f in right):
            raise RuntimeError(f'WindowStore inventory mismatch for {config}')
    print('SSD DATA READY — SHA256 VERIFIED:', verified_count, 'files')
    print('Local manifest:', LOCAL_SHA_MANIFEST)
    print('Drive manifest:', DRIVE_SHA_MANIFEST)
    if not VERIFY_DRIVE_SOURCE_CONTENT_ON_REUSE:
        print('NOTE: Drive CSV content is not rehashed on reuse; enable option if source may have changed')
else:
    print('Using Drive data directly. No SSD staging performed.')

## 7. Xác thực dataset & run identity trên đường dẫn đã chọn

Nếu dataset fingerprint/profile thay đổi vì đường dẫn mới, approval cũ không còn dùng được. Cell này fail-closed, không tự thay identity trong file approval.

In [ ]:
from colab.experiment_launcher import RunSpec, preflight
from colab.svm_run_authorization import collect_identity, expected_confirmation
SPEC=RunSpec(EXPERIMENT,CONFIG,RUN_NAME)
assert WINDOWS.is_dir(),f'WINDOWS missing: {WINDOWS}'
identity,identity_evidence=collect_identity(REPO,WINDOWS,SPEC)
print('=== VERIFIED RUN IDENTITY ===')
print(json.dumps(identity,indent=2))
print('Dataset fingerprint:',identity['dataset_fingerprint_sha256'])
print('Selected windows:',WINDOWS)
if (EXPERIMENT,CONFIG)==('baseline_svm','FW_512_FS_16'):
    previous_fp='dc8d8838c30ce09416f7df2b5915c1d36617066727ffbba62cf8b28423f70bde'
    if identity['dataset_fingerprint_sha256']!=previous_fp:
        print('NOTE: dataset fingerprint differs from previous Drive audit. New identity/technical approval required.')
CONFIRMATION=expected_confirmation(identity)
print('Required run-specific confirmation:',CONFIRMATION)
print('IDENTITY COLLECTION: PASS (not yet authorized)')

## 8. Kiểm tra Unit Test Evidence đã có (không chạy lại 69 tests)

Nếu Git thay đổi hoặc evidence không còn, hãy chạy unit tests theo quy trình Stage 7.3B rồi ghi nhận evidence mới. Không tự đánh dấu `passed=True`.

In [ ]:
if not UNIT_EVIDENCE.is_file():
    raise FileNotFoundError(f'Unit test evidence missing: {UNIT_EVIDENCE}')
unit=json.loads(UNIT_EVIDENCE.read_text(encoding='utf-8'))
checks={
    'passed':unit.get('passed') is True,
    'commit':unit.get('git_commit')==EXPECTED_COMMIT,
    'training_hash':unit.get('training_sha256')==sha256_file(REPO/'huyen_code/ciciomt24_training.py'),
    'adapter_hash':unit.get('adapter_test_sha256')==sha256_file(REPO/'colab/test_stage6_colab.py'),
}
print('UNIT EVIDENCE:',checks)
if not all(checks.values()): raise RuntimeError('Unit evidence is missing/stale; DO NOT TRAIN')
print('UNIT EVIDENCE: PASS')

## 8A. **Tùy chọn:** tạo Train-only technical probe khi cấu hình chưa có bằng chứng

Nếu đã có probe hợp lệ cho cấu hình hiện chọn, **bỏ qua cell này**. Với cấu hình mới, đổi `RUN_TRAIN_ONLY_PROBE=True` rồi chỉ chạy cell này một lần. Probe chỉ fit mẫu Train tối đa 8.192 hàng, **không** chạy Validation/Test, không phải training chính thức. Output được lưu Drive, không ghi đè. Chạy xong tiếp tục cell 9 để xác minh.

In [ ]:
RUN_TRAIN_ONLY_PROBE = False
PROBE_SIZE = 8192
if not RUN_TRAIN_ONLY_PROBE:
    print('Train-only probe not requested in this session')
else:
    from colab.dataset_validator import manifest_root_for
    manifest_root=manifest_root_for(WINDOWS,REPO)
    probe_script=REPO/'huyen_code/ciciomt24_svm_probe.py'
    if not probe_script.is_file(): raise FileNotFoundError(probe_script)
    probe_name=f'{RUN_NAME}_probe8192_technical.json'
    local_probe=Path('/content')/probe_name
    drive_probe=DRIVE_ROOT/'technical_probes'/probe_name
    if local_probe.exists() or drive_probe.exists():
        raise FileExistsError('Technical probe exists: do not overwrite; inspect existing evidence')
    cmd=[sys.executable,str(probe_script),'probe',
         '--repo',str(REPO),'--windows',str(WINDOWS),
         '--manifest-root',str(manifest_root),'--experiment',EXPERIMENT,
         '--config',CONFIG,'--output',str(local_probe),
         '--sizes',str(PROBE_SIZE),'--seed','42',
         '--timeout-per-size','300','--max-rss-gib','6']
    env=os.environ.copy()
    env['PYTHONPATH']=os.pathsep.join([str(REPO),str(REPO/'huyen_code'),env.get('PYTHONPATH','')])
    rc=subprocess.run(cmd,cwd=REPO,env=env,check=False).returncode
    if not local_probe.is_file(): raise RuntimeError(f'Probe output missing; exit={rc}')
    result=json.loads(local_probe.read_text(encoding='utf-8'))
    if rc!=0 or result.get('status')!='PASS_TECHNICAL_ONLY':
        raise RuntimeError('Probe incomplete/failed — no successful evidence saved to Drive')
    DRIVE_PROBE_DIR=drive_probe.parent
    DRIVE_PROBE_DIR.mkdir(parents=True,exist_ok=True)
    with local_probe.open('rb') as src, drive_probe.open('xb') as dst:
        shutil.copyfileobj(src,dst,length=8*1024*1024)
    if sha256_file(drive_probe)!=sha256_file(local_probe):
        raise RuntimeError('Technical probe Drive copy checksum mismatch')
    print('TECHNICAL-ONLY PROBE PASS:',drive_probe)
    print('SHA256:',sha256_file(drive_probe))
    print('OFFICIAL TRAINING: NOT STARTED')

## 9. Chọn technical evidence hợp lệ, **có fallback**

Nếu có file 64K ưu tiên và file đó còn hợp lệ thì chọn nó. Nếu thiếu/hỏng/không đúng identity, tự tìm trong `technical_probes` (bao gồm file 8.192 mẫu vừa được tạo ở bước 8A); mọi ứng viên đều phải qua `_validate_technical_evidence(EXACT_CONFIG_PROBE)`. Nếu không có file nào hợp lệ thì **dừng**, không tự cho phép huấn luyện.

In [ ]:
from colab.svm_run_authorization import _validate_technical_evidence
PROBE_DIR = DRIVE_ROOT / 'technical_probes'
# Để None => tìm tự động; muốn cố định một file thì điền tên file chính xác.
TECHNICAL_EVIDENCE_OVERRIDE = None
PREFERRED_EVIDENCE_NAME = (
    'baseline_svm_FW512_FS16_scaling64k_01.json'
    if (EXPERIMENT, CONFIG) == ('baseline_svm', 'FW_512_FS_16') else None
)

def observed_source_identity(data):
    source = data.get('source_hashes') if isinstance(data.get('source_hashes'), dict) else {}
    return {'experiment': data.get('experiment'), 'config': data.get('config'),
            'training_code_sha256': data.get('training_code_sha256') or source.get('training'),
            'window_summary_sha256': data.get('window_summary_sha256') or source.get('summary')}

def validate_probe_candidate(p):
    data = json.loads(p.read_text(encoding='utf-8'))
    observed = observed_source_identity(data)
    policy = {'technical_evidence': {'policy': 'EXACT_CONFIG_PROBE',
              'evidence_sha256': sha256_file(p), 'evidence_identity': observed}}
    result = _validate_technical_evidence(policy, p, identity)
    return data, policy['technical_evidence'], result

if not PROBE_DIR.is_dir():
    raise FileNotFoundError(f'No technical_probes directory: {PROBE_DIR}')
if TECHNICAL_EVIDENCE_OVERRIDE:
    # Explicit choice must be exact, no silent substitution.
    requested = Path(TECHNICAL_EVIDENCE_OVERRIDE)
    if requested.name != str(requested):
        raise ValueError('Specify only the evidence filename, not an external path')
    candidates = [PROBE_DIR / requested.name]
else:
    candidates = []
    if PREFERRED_EVIDENCE_NAME:
        candidates.append(PROBE_DIR / PREFERRED_EVIDENCE_NAME)
    # Deterministic fallback: include new probe from 8A and any other files in the directory.
    candidates.extend(sorted(PROBE_DIR.glob('*.json')))
    candidates = list(dict.fromkeys(candidates))

valid, errors = [], []
for p in candidates:
    if not p.is_file():
        errors.append((p.name, 'missing'))
        continue
    try:
        data, policy, result = validate_probe_candidate(p)
        valid.append((p, data, policy, result))
    except (OSError, ValueError, KeyError, PermissionError, TypeError) as exc:
        errors.append((p.name, f'{type(exc).__name__}: {exc}'))

if not valid:
    print('No valid exact-identity technical evidence. Errors:', errors[:12])
    raise RuntimeError('Technical evidence missing/invalid; do not create approval')

preferred = next((row for row in valid
                  if row[0].name == PREFERRED_EVIDENCE_NAME), None) if not TECHNICAL_EVIDENCE_OVERRIDE else None
if preferred:
    chosen = preferred
else:
    # Largest successful bounded sample, filename tie-break; never accept another config.
    chosen = max(valid, key=lambda row: (max(row[1].get('sizes') or [0]), row[0].name))
TECHNICAL_EVIDENCE, probe_data, technical_policy, technical_result = chosen
print('CHOSEN EVIDENCE:', TECHNICAL_EVIDENCE)
print('SHA256:', technical_result['sha256'])
print('Sizes:', probe_data.get('sizes'))
print('Valid candidates:', len(valid), '| Invalid/missing:', len(errors))
if errors:
    print('Rejected examples:', errors[:5])
print('EXACT_CONFIG_PROBE: PASS; OFFICIAL TRAINING: NOT STARTED')

## 10. Tạo bản approval **PENDING** để người dùng xem xét

Notebook không tự cấp quyền từ probe. Ngân sách ở đây là **giới hạn vận hành đề xuất** từ buổi kiểm tra Colab RAM cao, không phải dự báo thời gian/RSS thực tế. Với FS8 và MI-RFE, thời gian có thể vượt ngân sách; phải đánh giá riêng. Một `PENDING` draft sẽ nằm ở `approval_drafts`, không được supervisor chấp nhận.

In [ ]:
from colab.svm_run_authorization import SCHEMA, REQUIRED_BUDGET_FIELDS
budgets={
    'max_elapsed_seconds':14400,             # 4 giờ
    'max_tree_rss_bytes':28*GIB,            # 28 GiB
    'min_start_available_ram_bytes':42*GIB, # 42 GiB
    'min_runtime_available_ram_bytes':10*GIB,
    'min_free_disk_bytes':30*GIB,
    'poll_interval_seconds':10,
}
assert set(budgets)==set(REQUIRED_BUDGET_FIELDS)
if psutil.virtual_memory().available < budgets['min_start_available_ram_bytes']:
    print('WARNING: RAM available is already below proposed start requirement')
DRAFT=DRIVE_ROOT/'approval_drafts'/f'{RUN_NAME}_PENDING_REVIEW.json'
draft={
    'schema':SCHEMA,'decision':'PENDING_HUMAN_REVIEW','effective':False,
    'identity':identity,'confirmation':None,
    'scientific_contract_acknowledged':False,
    'test_is_single_final_evaluation_acknowledged':False,
    'capture_session_independence_not_established_acknowledged':False,
    'budgets':budgets,'technical_evidence':technical_policy,
}
DRAFT.parent.mkdir(parents=True,exist_ok=True)
if DRAFT.exists():
    old=json.loads(DRAFT.read_text(encoding='utf-8'))
    if old!=draft:
        print('An older different draft exists. Will not overwrite:',DRAFT)
        DRAFT=DRAFT.with_name(DRAFT.stem+'_SSD_REVIEW.json')
        if DRAFT.exists():
            if json.loads(DRAFT.read_text(encoding='utf-8'))!=draft:
                raise FileExistsError(f'A different draft already exists: {DRAFT}')
        else:
            DRAFT.write_text(json.dumps(draft,indent=2,ensure_ascii=False),encoding='utf-8')
else:
    DRAFT.write_text(json.dumps(draft,indent=2,ensure_ascii=False),encoding='utf-8')
print('Approval DRAFT:',DRAFT,'\nDecision:',draft['decision'],'Effective:',draft['effective'])
print('Limits:',json.dumps(budgets,indent=2))
print('Approval ACTIVE file:',APPROVAL,'Exists:',APPROVAL.exists())
print('NO AUTHORIZATION GRANTED')

## 11. **Tùy chọn:** human-reviewed approval, tạo bằng thao tác xác nhận trực tiếp trong Colab

**KHÔNG bật nếu chưa tự đánh giá và đồng ý:** nhãn scenario proxy, capture/session chưa chứng minh độc lập, Test chỉ đánh giá một lần, MI-RFE selection chỉ dùng Train/Val theo pipeline, ngân sách thời gian/RAM/disk. Phần này **chưa chạy training**. Khi `HUMAN_REVIEW_AND_CREATE_APPROVAL=True`, notebook hiển thị lại ngân sách, yêu cầu gõ **hai câu xác nhận**; chỉ khi khớp chính xác mới viết approval `APPROVED_STAGE73B` dạng file mới, không ghi đè. Trên phiên Colab khác, nếu file phê duyệt đã tồn tại, cell chỉ đối chiếu nội dung và báo; không tự tạo lại.

In [ ]:
HUMAN_REVIEW_AND_CREATE_APPROVAL = False  # Chỉ đổi TRUE sau khi em tự đồng ý toàn bộ rủi ro

if not HUMAN_REVIEW_AND_CREATE_APPROVAL:
    print('PENDING: human has not reviewed and approved this run. No approval file created.')
else:
    print('=== REVIEW BEFORE AUTHORIZATION ===')
    print('Run:',RUN_NAME,'\nSource:',identity['git_commit'],'\nDataset:',identity['dataset_fingerprint_sha256'])
    print('Technical evidence:',TECHNICAL_EVIDENCE,'\nProposed budgets:',json.dumps(budgets,indent=2))
    print('Scientific caveats: SCENARIO_PROXY; flow truth/capture independence not established.')
    print('Final TEST: at most once; do not tune using TEST.')
    print('Stopping at time/RAM/disk limit can leave an incomplete, nonresumable SVM run.')
    human_ack = input('Gõ CHÍNH XÁC "TÔI CHẤP NHẬN CÁC GIỚI HẠN KHOA HỌC VÀ TÀI NGUYÊN": ').strip()
    if human_ack != 'TÔI CHẤP NHẬN CÁC GIỚI HẠN KHOA HỌC VÀ TÀI NGUYÊN':
        raise PermissionError('Scientific/resource review not explicitly acknowledged')
    human_phrase = input('Gõ CHÍNH XÁC câu AUTHORIZE STAGE7.3B đã in ở Cell 7: ').strip()
    if human_phrase != CONFIRMATION:
        raise PermissionError('Exact per-run human authorization phrase differs')
    final_approval=dict(draft)
    final_approval.update(
        decision='APPROVED_STAGE73B',effective=True,confirmation=human_phrase,
        scientific_contract_acknowledged=True,
        test_is_single_final_evaluation_acknowledged=True,
        capture_session_independence_not_established_acknowledged=True,
    )
    from colab.svm_run_authorization import _validate_technical_evidence
    _validate_technical_evidence(final_approval, TECHNICAL_EVIDENCE,identity)
    APPROVAL.parent.mkdir(parents=True,exist_ok=True)
    if APPROVAL.exists():
        existing=json.loads(APPROVAL.read_text(encoding='utf-8'))
        if existing!=final_approval:
            raise FileExistsError('Conflicting approval file exists; NEVER overwrite')
        print('Existing exact same effective approval retained:',APPROVAL)
    else:
        with APPROVAL.open('x',encoding='utf-8') as f:
            json.dump(final_approval,f,indent=2,ensure_ascii=False)
        print('EXPLICIT HUMAN-REVIEWED APPROVAL CREATED:',APPROVAL)
    print('TRAINING: NOT STARTED')

## 12. Read-only authorization review trước khi chạy

`authorize_run()` thực hiện Preflight và so sánh **tất cả identity fields**; lỗi Stage 7.1 `svm_full_resource_gate_reviewed=False` được Stage 7.3B xử lý riêng. Không sửa các cờ này. Approval cho đường dẫn Drive **có thể không dùng được với SSD** nếu fingerprint/profile thay đổi.

In [ ]:
from colab.svm_run_authorization import authorize_run
AUTH_CONTEXT=None
if not APPROVAL.is_file():
    print('BLOCKED: effective approval file missing:',APPROVAL)
else:
    approved=json.loads(APPROVAL.read_text(encoding='utf-8'))
    if approved.get('decision')!='APPROVED_STAGE73B' or approved.get('effective') is not True:
        raise PermissionError('Approval is not effective')
    AUTH_CONTEXT=authorize_run(REPO,WINDOWS,LOCAL_RUNS,DRIVE_ROOT,SPEC,
                                APPROVAL,TECHNICAL_EVIDENCE,CONFIRMATION,
                                unit_test_evidence=unit)
    print('AUTHORIZED CONTEXT VALIDATED:',json.dumps(AUTH_CONTEXT.public_record(),indent=2))
    print('NO MODEL.FIT OR TEST PREDICTION PERFORMED')

## 13. Cổng chạy chính thức (từng run một, giám sát RSS/time/disk)

**Mặc định không chạy.** Sau khi approval thật sự có hiệu lực, để chạy đúng một SVM official: đặt cả hai cờ `True`. Lệnh gọi **supervisor của repository** (không gọi `SVC.fit()` tách rời). Quá trình sẽ huấn luyện theo pipeline gốc, thực hiện Test **một lần cuối**, lưu checkpoint/artifacts lên Drive. Không có resume mid-fit; nếu bị ngắt phải kiểm tra `.incomplete` trước khi tạo run mới. Thời gian thực tế có thể dài hơn ngân sách.

In [ ]:
START_OFFICIAL_SVM = False
I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION = False

if not (START_OFFICIAL_SVM and I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION):
    print('OFFICIAL SVM DISABLED — no training started')
else:
    if AUTH_CONTEXT is None:
        raise PermissionError('Cell 12 did not produce validated authorization context')
    if not APPROVAL.is_file() or not TECHNICAL_EVIDENCE.is_file():
        raise FileNotFoundError('Missing approval / technical evidence')
    if LOCAL_RUNS.joinpath(RUN_NAME).exists() or LOCAL_RUNS.joinpath(RUN_NAME+'.incomplete').exists():
        raise FileExistsError('An existing or incomplete local run blocks retry')
    cmd=[sys.executable,'-m','colab.svm_official_supervisor',
         '--repo',str(REPO),'--windows',str(WINDOWS),
         '--local-runs',str(LOCAL_RUNS),'--drive-root',str(DRIVE_ROOT),
         '--experiment',EXPERIMENT,'--config',CONFIG,'--run-name',RUN_NAME,
         '--approval',str(APPROVAL),
         '--technical-evidence',str(TECHNICAL_EVIDENCE),
         '--unit-test-evidence',str(UNIT_EVIDENCE),
         '--authorization-confirmation',CONFIRMATION]
    env=os.environ.copy()
    env['PYTHONPATH']=os.pathsep.join([str(REPO),str(REPO/'huyen_code'),env.get('PYTHONPATH','')])
    print('STARTING ONE OFFICIAL SVM RUN:',RUN_NAME,flush=True)
    finished=subprocess.run(cmd,cwd=REPO,env=env,check=False)
    print('SUPERVISOR EXIT CODE:',finished.returncode)
    if finished.returncode:
        raise RuntimeError('Supervisor did not exit cleanly. Inspect .incomplete artifacts; do not auto-retry')
    print('COMPLETE — review Drive experiment_results:',RUN_NAME)

## 14. Hậu kiểm kết quả chính thức (**metadata-only**, không predict Test)

Không coi thư mục tồn tại là PASS. Đọc `result.json`, `rows_by_split`, `test_evaluation_calls=1`, các sự kiện Test đúng một lần và SHA256 mọi artifact theo `artifact_manifest.json`. Đọc đúng báo cáo supervisor tại `DRIVE_ROOT / 'reports' / f'{RUN_NAME}_supervisor.json'`, chấp nhận `COMPLETED_VERIFIED` và đối chiếu `supervisor_stop_reason`. Nếu báo cáo không có hoặc không hợp lệ thì **không** báo PASS toàn bộ; có thể chỉ định đường dẫn bằng `SUPERVISOR_REPORT_OVERRIDE`.

In [ ]:
# CELL 14 — VERIFIED OFFICIAL RESULT REPORT, READ ONLY
# If supervisor report is not located: result/artifact PASS is only a partial verification.
SUPERVISOR_REPORT_OVERRIDE = None  # Optional exact path, when discovered from supervisor source
from colab.experiment_launcher import EXPECTED
import ast

RESULT_DIR = DRIVE_ROOT / 'experiment_results' / RUN_NAME
RESULT_FILE = RESULT_DIR / 'result.json'
ARTIFACT_MANIFEST = RESULT_DIR / 'artifact_manifest.json'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / f'{RUN_NAME}.incomplete'
INCOMPLETE_LOCAL = LOCAL_RUNS / f'{RUN_NAME}.incomplete'
INCOMPLETE_DRIVE = DRIVE_ROOT / 'experiment_results' / f'{RUN_NAME}.incomplete'

print('=== VERIFIED OFFICIAL RESULT ===')
print('Run:', RUN_NAME, 'Experiment:', EXPERIMENT, 'Config:', CONFIG)
print('Completed result directory:', RESULT_DIR)
print('Local incomplete:', INCOMPLETE_LOCAL.exists(), '| Drive incomplete:', INCOMPLETE_DRIVE.exists())
print('Checkpoint directory:', CHECKPOINT_DIR.exists(), '(non-resumable)')

if not RESULT_FILE.is_file() or not ARTIFACT_MANIFEST.is_file():
    print('RESULT STATUS:', 'INCOMPLETE_OR_FAILED' if (
        INCOMPLETE_LOCAL.exists() or INCOMPLETE_DRIVE.exists() or CHECKPOINT_DIR.exists()
    ) else 'NOT_COMPLETED')
    print('No Test rows read or predicted by this cell.')
else:
    result = json.loads(RESULT_FILE.read_text(encoding='utf-8'))
    manifest = json.loads(ARTIFACT_MANIFEST.read_text(encoding='utf-8'))
    expected_rows = EXPECTED[CONFIG]
    checks = {
        'result_status': result.get('status') in ('PASS', 'COMPLETED_WITH_CONVERGENCE_WARNING'),
        'result_experiment': result.get('experiment') == EXPERIMENT,
        'result_config': result.get('config') == CONFIG,
        'result_rows_by_split': result.get('rows_by_split') == expected_rows,
        'single_final_test_call': result.get('test_evaluation_calls') == 1,
        'manifest_experiment': manifest.get('experiment', (manifest.get('identity') or {}).get('experiment')) == EXPERIMENT,
        'manifest_config': manifest.get('config', (manifest.get('identity') or {}).get('config')) == CONFIG,
        'manifest_run_name': manifest.get('run_name', (manifest.get('identity') or {}).get('run_name')) == RUN_NAME,
    }
    required_names = {
        'preflight_audit.json', 'events.jsonl', 'progress.json',
        'selection_frozen.json', 'features.json', 'model.joblib',
        'final_scaler.joblib', 'result.json', 'authorization.json',
        'hyperparameters.json', 'feature_schema.json'
    }
    if EXPERIMENT == 'mi_rfe_svm':
        required_names.update({
            'mi_feature_scores.csv', 'mi_top100.json',
            'rfe_feature_ranking.csv', 'k_search_results.json'
        })
    files = manifest.get('files')
    checks['manifest_has_file_map'] = isinstance(files, dict)
    if not isinstance(files, dict):
        files = {}
    checks['all_required_artifacts_manifested'] = required_names.issubset(files)
    checks['no_duplicate_or_unsafe_artifact_names'] = all(
        isinstance(name, str) and Path(name).name == name and name not in ('.', '..')
        for name in files
    )
    artifact_failures = []
    if checks['no_duplicate_or_unsafe_artifact_names']:
        for name, meta in files.items():
            p = RESULT_DIR / name
            if (not isinstance(meta, dict) or not p.is_file()
                    or type(meta.get('bytes')) is not int
                    or meta.get('bytes') != p.stat().st_size
                    or not isinstance(meta.get('sha256'), str)
                    or sha256_file(p).casefold() != meta['sha256'].casefold()):
                artifact_failures.append(name)
    else:
        artifact_failures.append('INVALID_MANIFEST_FILE_NAME')
    checks['all_manifest_sha256_match'] = not artifact_failures and bool(files)

    event_file = RESULT_DIR / 'events.jsonl'
    event_names = []
    event_parse_error = None
    try:
        for line in event_file.read_text(encoding='utf-8').splitlines():
            if line.strip():
                item = json.loads(line)
                if not isinstance(item, dict):
                    raise ValueError('Event must be a JSON object')
                event_names.append(item.get('event'))
    except (OSError, ValueError, TypeError) as exc:
        event_parse_error = f'{type(exc).__name__}: {exc}'
    checks['test_event_started_once'] = event_names.count('test_evaluation_started') == 1
    checks['test_event_completed_once'] = event_names.count('test_evaluation_completed') == 1
    expected_order = ['selection_frozen', 'final_refit_completed',
                      'test_evaluation_started', 'test_evaluation_completed', 'run_validated']
    checks['scientific_event_order'] = (
        all(event in event_names for event in expected_order)
        and [event_names.index(e) for e in expected_order]
             == sorted(event_names.index(e) for e in expected_order)
    )
    checks['event_json_parsed'] = event_parse_error is None

    # Supervisor reports can vary by Stage 7.3B revision. Inspect only plausible
    # run-specific JSONs and DO NOT confuse checkpoint state with supervisor PASS.
    supervisor_candidates = [
        DRIVE_ROOT / 'reports' / f'{RUN_NAME}_supervisor.json',  # Actual Stage 7.3B supervisor output
        Path(SUPERVISOR_REPORT_OVERRIDE) if SUPERVISOR_REPORT_OVERRIDE else LOCAL_RUNS / '__no_manual_supervisor_report__.json',
        LOCAL_RUNS / f'{RUN_NAME}.supervisor.json',
        LOCAL_RUNS / f'{RUN_NAME}.supervisor_report.json',
        LOCAL_RUNS / f'{RUN_NAME}.incomplete.supervisor.json',
        DRIVE_ROOT / 'supervisor_reports' / f'{RUN_NAME}.json',
        DRIVE_ROOT / 'supervisor' / f'{RUN_NAME}.json',
        DRIVE_ROOT / 'supervision' / f'{RUN_NAME}.json',
    ]
    for folder in [LOCAL_RUNS, DRIVE_ROOT / 'reports', DRIVE_ROOT / 'supervisor_reports',
                   DRIVE_ROOT / 'supervision', DRIVE_ROOT / 'supervisor']:
        if folder.is_dir():
            supervisor_candidates.extend(sorted(folder.glob(f'*{RUN_NAME}*.json')))
    supervisor_candidates = list(dict.fromkeys(supervisor_candidates))
    supervisor_records = []
    for p in supervisor_candidates:
        if p.is_file():
            try:
                data = json.loads(p.read_text(encoding='utf-8'))
                if isinstance(data, dict) and (
                    data.get('identity', {}).get('run_name') == RUN_NAME
                    if isinstance(data.get('identity'), dict)
                    else data.get('run_name') == RUN_NAME
                ):
                    supervisor_records.append((str(p), data))
            except (OSError, ValueError):
                pass
    print('\nRESULT SCIENTIFIC STATUS:', result.get('status'))
    print('ROWS OBSERVED:', result.get('rows_by_split'))
    print('ROWS EXPECTED:', expected_rows)
    print('TEST EVALUATION CALLS:', result.get('test_evaluation_calls'))
    print('ARTIFACTS HASHED:', len(files))
    if artifact_failures:
        print('FAILED ARTIFACTS:', artifact_failures)
    if event_parse_error:
        print('EVENT PARSE ERROR:', event_parse_error)
    supervisor_valid = False
    if supervisor_records:
        for p, data in supervisor_records:
            print('SUPERVISOR REPORT:', p)
            status = data.get('status')
            stop = data.get('supervisor_stop_reason', data.get('stop_reason'))
            code = data.get('worker_returncode', data.get('returncode'))
            print('  Status:', status, '| Stop reason:', stop,
                  '| Worker return code:', code)
            if isinstance(data.get('identity'), dict):
                same_run = all(data['identity'].get(k) == v for k, v in (
                    ('run_name', RUN_NAME), ('experiment', EXPERIMENT), ('config', CONFIG)))
            else:
                same_run = data.get('run_name') == RUN_NAME and all(
                    data.get(k, v) == v for k, v in
                    (('experiment', EXPERIMENT), ('config', CONFIG)))
            reported_success = status in (
                'PASS', 'COMPLETED', 'COMPLETED_WITH_CONVERGENCE_WARNING',
                'SUCCESS', 'PASS_OFFICIAL', 'PASS_STAGE73B_COMPLETED',
                'COMPLETED_VERIFIED'
            )
            if same_run and stop is None and code == 0 and reported_success:
                supervisor_valid = True
    else:
        print('SUPERVISOR REPORT: not located; supervisor completion is NOT independently verified')
        print('If report exists elsewhere, set SUPERVISOR_REPORT_OVERRIDE to its exact path')
    print('SUPERVISOR CHECK:', 'PASS' if supervisor_valid else 'UNVERIFIED')
    print('\nCHECKS:')
    for name, ok in checks.items():
        print(f'  {name}:', 'PASS' if ok else 'FAIL')
    artifacts_pass = all(checks.values())
    label = ('PASS' if artifacts_pass and supervisor_valid else
             'PASS_ARTIFACTS_ONLY_SUPERVISOR_UNVERIFIED' if artifacts_pass else 'FAIL')
    print('FINAL VERIFICATION:', label)
    print('No training or Test prediction was performed by this cell.')

## 15. Checklist 4 cấu hình

Một notebook nhưng **mỗi lượt đúng một** `(EXPERIMENT, CONFIG, RUN_NAME)`:

| Config | Final Train+Val windows | Training paths |
|---|---:|---|
| `FW_256_FS_16` | 898.846 | Full SVC / MI–RFE SVC |
| `FW_512_FS_16` | 897.932 | Full SVC / MI–RFE SVC |
| `FW_256_FS_8` | 1.797.660 | Full SVC / MI–RFE SVC |
| `FW_512_FS_8` | 1.795.836 | Full SVC / MI–RFE SVC |

**Không đổi:** SVM RBF, `C=10`, `max_iter=20000`, `probability` theo nhánh, `StandardScaler`, MI–RFE, nguyên tắc split/leakage. **Không đo Test nhiều lần để tuning.**

**Chỉ dùng kết quả khoa học chính thức khi** worker hoàn thành và xác minh đủ artifact. `PENDING_HUMAN_REVIEW`, `PASS_TECHNICAL_ONLY`, `PASS_PRELIMINARY` không tương đương kết quả train chính thức.